# Step 1 playground: synthetic data in DuckDB
Run top to bottom. Kernel: the project `.venv` (needs `ipykernel`).

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "playground" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
import duckdb, pandas as pd
pd.set_option("display.width", 160)
DB = "data/warehouse.duckdb"
print("project root:", ROOT)

## 1. (Optional) Regenerate the data
Deletes and recreates the file. Same seed = same data.

In [ ]:
from jobs.data_gen.generate import write
# standalone playground: keep candidates/rules in DuckDB too (in the real flow they arrive via sap_ingest)
write(DB, seed=42, with_sap_tables=True)

## 2. Tables and row counts

In [ ]:
con = duckdb.connect(DB, read_only=True)
tables = [r[0] for r in con.sql("SHOW TABLES").fetchall()]
pd.DataFrame({t: [con.sql(f"SELECT count(*) FROM {t}").fetchone()[0]] for t in tables}, index=["rows"]).T

## 3. Peek at each table

In [ ]:
for t in tables:
    print(f"--- {t}")
    display(con.sql(f"SELECT * FROM {t} LIMIT 5").df())

## 4. Planted issues (seed_manifest)

In [ ]:
con.sql("SELECT * FROM seed_manifest").df()

## 5. Candidates joined with rules (NULL floor = missing rule)

In [ ]:
con.sql('''
SELECT c.sku, c.pack_type, c.pack_qty, c.region, c.shelf_price, c.week_no, c.current_price, r.price_floor, r.max_markdown_pct, r.max_clearance_weeks, p.cost
FROM clearance_candidates c
LEFT JOIN business_rules r USING (week, sku, pack_qty)
LEFT JOIN products p USING (sku, pack_qty)
WHERE c.week = '2026-W39' AND c.region = 'VIC' ORDER BY c.sku, c.pack_qty
''').df()

## 6. Check each planted issue shows up in the data

In [ ]:
print("Missing rule:");        display(con.sql("SELECT c.sku, c.pack_qty FROM clearance_candidates c LEFT JOIN business_rules r USING (week, sku, pack_qty) WHERE c.week='2026-W39' AND r.sku IS NULL AND c.region='VIC'").df())
print("Floor above price:");   display(con.sql("SELECT c.sku, c.pack_qty, c.current_price, r.price_floor FROM clearance_candidates c JOIN business_rules r USING (week, sku, pack_qty) WHERE c.week='2026-W39' AND r.price_floor > c.current_price AND c.region='VIC'").df())
print("Zero inventory:");      display(con.sql("SELECT * FROM inventory WHERE week='2026-W39' AND on_hand = 0").df())
print("No product master:");   display(con.sql("SELECT c.sku, c.pack_qty FROM clearance_candidates c LEFT JOIN products p USING (sku, pack_qty) WHERE c.week='2026-W39' AND p.sku IS NULL AND c.region='VIC'").df())

## 6b. All weeks are stored
History W27-W33, published clearance W34-W38, open week W39.

In [ ]:
display(con.sql("SELECT * FROM weeks").df())
con.sql('''SELECT week, count(DISTINCT sku) AS skus, count(*) AS rows FROM clearance_candidates GROUP BY 1 ORDER BY 1''').df()

## 6c. Constraint: price this week <= last week (same SKU, same region)
Must return 0 violations. W33 is the regular price, so a SKU's first clearance week is compared with it.

In [ ]:
viol = con.sql('''
SELECT * FROM (
  SELECT week, sku, pack_qty, region, price, lag(price) OVER (PARTITION BY sku, pack_qty, region ORDER BY week) AS prev
  FROM price_history WHERE week >= '2026-W33')
WHERE prev IS NOT NULL AND price > prev + 1e-9''').df()
print("violations:", len(viol)); viol.head()

In [ ]:
# one item's markdown path (region VIC), Single pack
sku = con.sql("SELECT sku FROM clearance_candidates WHERE week='2026-W34' AND pack_qty=1 LIMIT 1").fetchone()[0]
con.sql(f"SELECT week, sku, pack_type, pack_qty, price, is_clearance FROM price_history WHERE sku='{sku}' AND pack_qty=1 AND region='VIC' ORDER BY week").df()

## 6d. Multipacks
Different pack sizes share the same `sku`; `pack_type` (Single / Multipack) and `pack_qty` (units in the pack) tell them apart. Constraint: pack price / pack_qty >= the Single's price, same week and region. Must return 0 violations.

In [ ]:
display(con.sql("SELECT pack_type, pack_qty, count(*) AS rows, count(DISTINCT sku) AS distinct_sku_ids FROM products GROUP BY 1, 2 ORDER BY 2").df())
# the same sku id appears once per pack size
example = con.sql("SELECT sku FROM products WHERE pack_qty = 12 LIMIT 1").fetchone()[0]
display(con.sql(f"SELECT sku, pack_type, pack_qty, name, shelf_price, cost FROM products WHERE sku = '{example}' ORDER BY pack_qty").df())
viol = con.sql('''
SELECT pk.week, pk.sku, pk.pack_qty, pk.region, pk.price/pk.pack_qty AS unit_price, sg.price AS single_price
FROM price_history pk
JOIN price_history sg ON sg.sku = pk.sku AND sg.pack_qty = 1 AND sg.region = pk.region AND sg.week = pk.week
WHERE pk.pack_qty > 1 AND pk.week >= '2026-W34' AND pk.price/pk.pack_qty < sg.price - 1e-9''').df()
print("violations:", len(viol))

In [ ]:
# one item across its pack sizes: per-item price by week (region VIC)
sku = con.sql("SELECT sku FROM price_history WHERE pack_qty = 6 AND is_clearance LIMIT 1").fetchone()[0]
con.sql(f'''
SELECT week, pack_type, pack_qty, price AS pack_price, round(price / pack_qty, 2) AS unit_price
FROM price_history WHERE sku = '{sku}' AND region = 'VIC' AND week >= '2026-W33'
ORDER BY week, pack_qty''').df()

## 6e. Floors and max markdown respected in clearance weeks (0 violations each)

In [ ]:
print("below floor:", con.sql('''SELECT count(*) FROM price_history h JOIN business_rules r ON r.sku=h.sku AND r.pack_qty=h.pack_qty AND r.week=h.week WHERE h.is_clearance AND h.price < r.price_floor - 1e-9''').fetchone()[0])
print("beyond max markdown:", con.sql('''SELECT count(*) FROM price_history h JOIN business_rules r ON r.sku=h.sku AND r.pack_qty=h.pack_qty AND r.week=h.week JOIN products p ON p.sku=h.sku AND p.pack_qty=h.pack_qty WHERE h.is_clearance AND h.price < p.shelf_price*(1-r.max_markdown_pct) - 0.01''').fetchone()[0])

## 6f. shelf_price and week_no
`shelf_price` is the full price before any markdown. `week_no` is the nth consecutive week an item has been on the clearance list (1 = first week). Hard rules: price never below 50% off shelf price (the floor wins if higher); an item is listed for at most `max_clearance_weeks` (20) weeks, then SAP stops listing it but its history stays.

In [ ]:
# week_no runs 1, 2, 3 ... for each item (region VIC shown)
con.sql('''
SELECT sku, pack_type, pack_qty, week, week_no, shelf_price, price, round(price / shelf_price, 3) AS price_vs_shelf
FROM price_history WHERE is_clearance AND region = 'VIC' ORDER BY sku, pack_qty, week LIMIT 15''').df()

In [ ]:
# the lowest price relative to shelf price must be >= 0.5 (0 rows below 50% off)
print("rows below 50% of shelf:", con.sql("SELECT count(*) FROM price_history WHERE is_clearance AND price < shelf_price * 0.5 - 0.01").fetchone()[0])
print("lowest price / shelf   :", con.sql("SELECT round(min(price / shelf_price), 3) FROM price_history WHERE is_clearance").fetchone()[0])
print("max week_no on the W39 list:", con.sql("SELECT max(week_no) FROM clearance_candidates WHERE week = '2026-W39'").fetchone()[0], "(limit 20)")

In [ ]:
# removal after the limit: real data spans only W34-W39 so nobody reaches 20 weeks.
# Rebuild in memory with a limit of 3 weeks to see items dropped from the list but kept in history.
from jobs.data_gen.generate import build
d = build(42, max_clearance_weeks=3)
c = d["clearance_candidates"]; c = c[c.sku != "999999"]
per_item = c.drop_duplicates(["week", "sku", "pack_qty"]).groupby(["sku", "pack_qty"]).agg(first=("week", "min"), last=("week", "max"), weeks_listed=("week_no", "max"))
print("max week_no:", c.week_no.max())
display(per_item[per_item.weeks_listed == 3].head())
sku = per_item[per_item.weeks_listed == 3].index[0]
print("history for", sku, "still has", len(d["price_history"][(d["price_history"].sku == sku[0]) & (d["price_history"].pack_qty == sku[1])]), "price_history rows and",
      len(d["sales"][(d["sales"].sku == sku[0]) & (d["sales"].pack_qty == sku[1])]), "sales rows")

## 7. Sales behaviour: price vs units for one SKU
Lower weekly price should give higher units (elasticity is negative).

In [ ]:
wk = con.sql('''
SELECT date_trunc('week', sale_date) AS wk, sum(units) AS units, round(avg(price),2) AS avg_price
FROM sales WHERE sku = '100000' GROUP BY 1 ORDER BY 1
''').df()
display(wk)
wk.plot(x="avg_price", y="units", kind="scatter", title="SKU 100000: weekly price vs units")

## 8. Through the WarehouseClient (what services will use)

In [ ]:
from shared.warehouse import get_warehouse
wh = get_warehouse()          # reads env vars; defaults to DuckDB read-only
cands = wh.get_candidates("2026-W39")
print("candidate rows:", len(cands), "| distinct SKUs:", len({c['sku'] for c in cands}))
print("rules:", len(wh.get_rules("2026-W39")))
skus = sorted({c["sku"] for c in cands})[:3]
print("sales rows (VIC):", len(wh.get_sales_history(skus, "VIC")))
print("inventory rows:", len(wh.get_inventory(skus, "2026-W39")))
pd.DataFrame(cands).head()

## 9. Determinism check
Same seed must give identical data.

In [ ]:
import tempfile
a, b = tempfile.mktemp(suffix=".duckdb"), tempfile.mktemp(suffix=".duckdb")
write(a, seed=7, with_sap_tables=True); write(b, seed=7, with_sap_tables=True)
from shared.warehouse import DuckDBWarehouse
print("identical:", DuckDBWarehouse(a).get_rules("2026-W39") == DuckDBWarehouse(b).get_rules("2026-W39"))

In [ ]:
con.close()